In [ ]:
import pandas as pd
import glob
import os
import logging
import matplotlib.pyplot as plt
import xarray as xr
from datetime import datetime
import shutil
import numpy as np

# ==============================
# 1. KONFIGURASI
# ==============================
DATA_DIR  = 'data' 
RAW_DIR   = os.path.join(DATA_DIR,'00.Raw_Dataset')
QC_DIR    = os.path.join(DATA_DIR,'01.QC_Dataset_Level_01')
FINAL_DIR = os.path.join(DATA_DIR,'04.Dataset_Final')

# if os.path.exists(QC_DIR):
#     shutil.rmtree(QC_DIR)
# os.makedirs(QC_DIR, exist_ok=True)

PARAMS = [
    # 'TEMPERATURE_AVG_C',
    # 'TEMP_24H_TN_C',
    # 'TEMP_24H_TX_C',
    'RAINFALL_24H_MM'
]

AVG_COL   = 'TEMPERATURE_AVG_C'
MIN_COL   = 'TEMP_24H_TN_C'
MAX_COL   = 'TEMP_24H_TX_C'
TEMP_COLS = [AVG_COL, MIN_COL, MAX_COL]

ID_COL   = 'WMO_ID'
TIME_COL = 'DATA_TIMESTAMP'

START_DATE = '1981-01-01'
END_DATE   = datetime.now().strftime('%Y-%m-%d')

PHYSICAL_BOUNDS = {
    'TEMPERATURE_AVG_C': (15, 40),
    'TEMP_24H_TN_C':     (12, 28),
    'TEMP_24H_TX_C':     (20, 42),
    'RAINFALL_24H_MM':   (0, 600)
}

ABRUPT_THRESHOLDS = {
    'TEMPERATURE_AVG_C': 2.5,
    'TEMP_24H_TN_C':     2.5,
    'TEMP_24H_TX_C':     2.5,
    'RAINFALL_24H_MM':   250,
}



In [8]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==============================
# 2. FUNGSI BANTU
# ==============================

def get_latest_raw_file(raw_dir, pattern="02.BMKGSOFT_VIEW_FKLIM_DAILY_1991-2024_UPDATED_*.csv"):
    files = glob.glob(os.path.join(raw_dir, pattern))
    if not files:
        raise FileNotFoundError(f"Tidak ada file ditemukan di {raw_dir}")
    return sorted(files)[-1]

def create_qc_dir(qc_dir, param):
    main_output_dir = os.path.join(qc_dir, param)
    if os.path.exists(main_output_dir ):
        shutil.rmtree(main_output_dir )
    os.makedirs(main_output_dir , exist_ok=True)
    steps = [
        '00.80percent',
        '01.DuplicatesRemoved',
        '02.ConsistencyCheck',
        '03.RangeCheck',
        '04.AbruptChangesAdjusted'
    ]
    step_dirs = {
        step: {
            'plot': os.path.join(main_output_dir, step, 'plots'),
            'netcdf': os.path.join(main_output_dir, step, 'netcdf')
        }
        for step in steps
    }
    summary_dir = os.path.join(main_output_dir, '05.Summary')
    adjusted_dir = os.path.join(main_output_dir, '06.Adjusted')
    for d in [main_output_dir, adjusted_dir, summary_dir]:
        os.makedirs(d, exist_ok=True)
    for step in step_dirs.values():
        os.makedirs(step['plot'], exist_ok=True)
        os.makedirs(step['netcdf'], exist_ok=True)
    return main_output_dir, adjusted_dir, summary_dir, step_dirs

def save_station_plot_and_netcdf(df, station_id, time_col, param, plot_dir, netcdf_dir, title_suffix, qc_col=None):
    sdf = df[df[ID_COL] == station_id].copy()
    if sdf.empty:
        return
    plt.figure(figsize=(10, 4))
    if 'RAINFALL' in param:
        color = 'c'
        ylabel = 'Curah Hujan (mm)'
    else:
        color = 'r' if 'AVG' in param else ('b' if 'MIN' in param else 'g')
        ylabel = 'Temperature (°C)'
    
    col_to_plot = f'RAW_{param}'
    plt.plot(sdf[time_col], sdf[col_to_plot], color + '-', alpha=0.7, label='RAW')
    if qc_col and qc_col in sdf.columns:
        plt.plot(sdf[time_col], sdf[qc_col], 'k-', alpha=0.6, label='QC')
        plt.legend()
    plt.title(f'Station {station_id} – {title_suffix}')
    plt.xlabel('Date')
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.tight_layout()
    safe_name = title_suffix.replace(" ", "_").replace("/", "_").lower()
    plot_subdir = os.path.join(plot_dir, title_suffix.split(' ')[-1])
    os.makedirs(plot_subdir, exist_ok=True)
    plt.savefig(os.path.join(plot_subdir, f'Station_{station_id}_{safe_name}.png'), dpi=300)
    plt.close()
    
    try:
        nc_subdir = os.path.join(netcdf_dir, title_suffix.split(' ')[-1])
        os.makedirs(nc_subdir, exist_ok=True)
        sdf.set_index(time_col).to_xarray().to_netcdf(
            os.path.join(nc_subdir, f'Station_{station_id}_{safe_name}.nc')
        )
    except Exception as e:
        logging.warning(f"Gagal simpan NetCDF stasiun {station_id}: {e}")

def check_duplicate_date(df, time_col, id_col, start_date, end_date):
    df_full = df[(df[time_col] >= start_date) & (df[time_col] <= end_date)].copy()
    initial_rows = len(df_full)
    df_full = df_full.drop_duplicates(subset=[id_col, time_col], keep='first').reset_index(drop=True)
    logging.info(f"Duplikat berdasarkan waktu dihapus: {initial_rows - len(df_full)} baris")
    total_days_1981 = (pd.to_datetime(END_DATE) - pd.to_datetime('1981-01-01')).days + 1
    total_days_1991 = (pd.to_datetime(END_DATE) - pd.to_datetime('1991-01-01')).days + 1
    return df_full, total_days_1981, total_days_1991

def check_parameter_data(df, param_list):
    missing_cols = [p for p in param_list if p not in df.columns]
    if missing_cols:
        logging.warning(f"Kolom berikut tidak ditemukan: {missing_cols}")
        for col in missing_cols:
            df[col] = pd.NA
    else:
        logging.info(f"Kolom berikut ditemukan: {param_list}")

def check_availability(df, param, time_col, id_col, baseline_key, total_days_1981, total_days_1991):
    now = datetime.now().year
    if baseline_key == '1981':
        tahun = '1981'
        total_days = total_days_1981
        label = f'1981_{now}'
    elif baseline_key == '1991':
        tahun = '1991'
        total_days = total_days_1991
        label = f'1991_{now}'
    else:
        raise ValueError("Baseline harus '1981' atau '1991'")
    
    avail = (
        df[df[time_col] >= tahun].groupby(id_col)[f'RAW_{param}']
        .agg(**{f'AVAIL_{param}_{label}': lambda x: (x.notna().sum() / total_days) * 100})
        .reset_index()
    )
    avail[f'80%_{param}_{baseline_key}'] = avail[f'AVAIL_{param}_{label}'] >= 80
    valid_stations = avail[avail[f'80%_{param}_{baseline_key}']][id_col].tolist()
    logging.info(f"[{label}] Jumlah stasiun: {len(avail)} → {len(valid_stations)} lolos (≥80%)")
    return avail, valid_stations, label

def save_qc_step(df, param, valid_1991, valid_1981, adj_dir, step_dir, prefix, title_suffix):
    qc_col = f'QC_{param}'
    for baseline, valid_list in [('1991', valid_1991), ('1981', valid_1981)]:
        start = '1981-01-01' if baseline == '1981' else '1991-01-01'
        subset = df[df[ID_COL].isin(valid_list)].copy()
        subset = subset[subset[TIME_COL] >= pd.to_datetime(start)]
        
        path = os.path.join(adj_dir, f'{prefix}_{baseline}.csv')
        subset.to_csv(path, index=False)
        logging.info(f"Subset ≥80% (baseline {baseline}) disimpan: {path}")

        for sid in subset[ID_COL].unique():
            save_station_plot_and_netcdf(
                subset, sid, TIME_COL, param,
                step_dir['plot'], step_dir['netcdf'],
                f"{title_suffix} {baseline}",
                qc_col=qc_col
            )

def check_consistency(df, param, avg_col, min_col, max_col, id_col, time_col, sum_dir):
    df = df.copy().sort_values([id_col, time_col])
    
    for base_col in [avg_col, min_col, max_col]:
        qc_name = f'QC_{base_col}'
        raw_name = f'RAW_{base_col}'
        if raw_name not in df.columns:
            logging.warning(f"Kolom {raw_name} tidak ditemukan. Lewati consistency check.")
            return df
        if qc_name not in df.columns:
            df[qc_name] = df[raw_name].copy()
    
    if param not in [avg_col, min_col, max_col]:
        return df

    qc_avg = f'QC_{avg_col}'
    qc_min = f'QC_{min_col}'
    qc_max = f'QC_{max_col}'

    complete = df[qc_avg].notna() & df[qc_min].notna() & df[qc_max].notna()
    inconsistent = ((df[qc_avg] < df[qc_min]) | (df[qc_avg] > df[qc_max])) & complete

    df['TEMP_CONSISTENCY_FLAG'] = inconsistent.astype(int)
    df.loc[inconsistent, qc_avg] = pd.NA

    df[qc_avg] = df.groupby(id_col)[qc_avg].transform(
        lambda g: g.interpolate(method='linear', limit_direction='both')
    )

    to_estimate = df[qc_avg].isna() & df[qc_min].notna() & df[qc_max].notna()
    df.loc[to_estimate, qc_avg] = (df.loc[to_estimate, qc_min] + df.loc[to_estimate, qc_max]) / 2.0

    df[f'QC_{param}'] = df[qc_avg] if param == avg_col else (
                        df[qc_min] if param == min_col else df[qc_max])

    summary = df.groupby(id_col)['TEMP_CONSISTENCY_FLAG'].sum().reset_index()
    summary.to_csv(os.path.join(sum_dir, '02.Summary_betweenTnTx.csv'), index=False)
    return df

def check_range(df, param, bounds, id_col, sum_dir):
    df = df.copy()
    qc_col = f'QC_{param}'
    if qc_col not in df.columns or param not in bounds:
        return df

    min_val, max_val = bounds[param]
    flag = (df[qc_col] < min_val) | (df[qc_col] > max_val)
    df[f'{param}_range_check_flag'] = flag.astype(int)

    if param == 'RAINFALL_24H_MM':
        # 🔑 HANYA UNTUK HUJAN: jangan interpolasi!
        # Hanya set nilai invalid ke NaN, TIDAK ISI LAGI
        invalid = (df[qc_col] < 0) | (df[qc_col] > max_val)
        df.loc[invalid, qc_col] = pd.NA
        # ❌ TIDAK ADA INTERPOLASI ATAU fillna(0.0)
    else:
        # Untuk suhu: lakukan seperti biasa
        df.loc[flag, qc_col] = pd.NA
        df[qc_col] = df.groupby(id_col)[qc_col].transform(
            lambda g: g.interpolate(method='linear', limit_direction='both')
        )

    summary = df.groupby(id_col)[f'{param}_range_check_flag'].sum().reset_index()
    summary.to_csv(os.path.join(sum_dir, '03.Summary_range_check.csv'), index=False)
    return df

def adjust_for_abrupt_changes(df, param, threshold, id_col, time_col):
    if param not in ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C']:
        return df.copy()
    
    df = df.copy().sort_values([id_col, time_col])
    qc_col = f'QC_{param}'
    if qc_col not in df.columns:
        return df

    flag_col = f'abrupt_change_flag_{param}'
    df[flag_col] = 0

    def _flag(group):
        group = group.sort_values(time_col)
        diffs = group[qc_col].diff().abs()
        flags = (diffs > threshold) & group[qc_col].notna()
        flags.iloc[0] = False
        return flags.astype(int)

    df[flag_col] = df.groupby(id_col, group_keys=False).apply(_flag).values
    df.loc[df[flag_col] == 1, qc_col] = pd.NA
    df[qc_col] = df.groupby(id_col)[qc_col].transform(
        lambda g: g.interpolate(method='linear', limit_direction='both')
    )
    return df

# =================================================================
# MEMULAI PIPELINE QC
# =================================================================
logging.info("Memulai pipeline QC untuk parameter suhu dan curah hujan...")
raw_file = get_latest_raw_file(RAW_DIR)
logging.info(f"Loading raw data from: {raw_file}")

na_values = ["", " ", "NA", "N/A", "-", "null", "NULL", "None"]
raw_df = pd.read_csv(raw_file, low_memory=False, na_values=na_values, keep_default_na=True)
raw_df[TIME_COL] = pd.to_datetime(raw_df[TIME_COL], errors='coerce')
check_parameter_data(raw_df, PARAMS)

# Buat semua RAW_... sekaligus
for p in PARAMS:
    raw_df[f'RAW_{p}'] = raw_df[p].copy()

# Langkah 0: Hapus duplikat berdasarkan waktu
raw_df_clean00, total_days_1981, total_days_1991 = check_duplicate_date(raw_df, TIME_COL, ID_COL, START_DATE, END_DATE)

DF_FINAL_DICT = {}

for param in PARAMS:
    logging.info(f"\n{'='*60}")
    logging.info(f"Memproses parameter: {param}")
    logging.info(f"{'='*60}")

    main_out, adj_dir, sum_dir, step_dirs = create_qc_dir(QC_DIR, param)

    df_avail = raw_df_clean00.copy()
    df_avail[f'QC_{param}'] = df_avail[f'RAW_{param}'].copy()

    if param in TEMP_COLS:
        for temp_col in TEMP_COLS:
            if f'QC_{temp_col}' not in df_avail.columns:
                df_avail[f'QC_{temp_col}'] = df_avail[f'RAW_{temp_col}'].copy()

    # Langkah 1: Kelengkapan data (80%)
    logging.info(f"Langkah 1: Menghitung kelengkapan untuk baseline 1981 dan 1991...")
    avail_1991, valid_1991, _ = check_availability(df_avail, param, TIME_COL, ID_COL, '1991', total_days_1981, total_days_1991)
    avail_1981, valid_1981, _ = check_availability(df_avail, param, TIME_COL, ID_COL, '1981', total_days_1981, total_days_1991)
    
    avail_combined = avail_1991.merge(avail_1981, on=ID_COL, how='outer')
    avail_combined.to_csv(os.path.join(sum_dir, '00.Summary_80percent.csv'), index=False)
    
    df_avail[f'80PCT_1991_{param}'] = df_avail[ID_COL].isin(valid_1991).astype(int)
    df_avail[f'80PCT_1981_{param}'] = df_avail[ID_COL].isin(valid_1981).astype(int)

    for baseline, valid_list in [('1991', valid_1991), ('1981', valid_1981)]:
        df_subset00 = df_avail[df_avail[ID_COL].isin(valid_list)].copy()
        start_baseline = '1981-01-01' if baseline == '1981' else '1991-01-01'
        df_subset00 = df_subset00[df_subset00[TIME_COL] >= pd.to_datetime(start_baseline)]
        output_path = os.path.join(adj_dir, f'00.filtered_80percent_{baseline}.csv')
        df_subset00.to_csv(output_path, index=False)
        for sid in df_subset00[ID_COL].unique():
            save_station_plot_and_netcdf(
                df_subset00, sid, TIME_COL, param,
                step_dirs['00.80percent']['plot'],
                step_dirs['00.80percent']['netcdf'],
                f"Original Time Series Baseline {baseline}",
                qc_col=None
            )

    # Langkah 2: DVAT
    logging.info(f"Langkah 2: Removing duplicates for {param}...")
    meta_cols = {'NAME', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 'PROVINSI', 'KABUPATEN', 'ELEVATION', 'WIND_DIR_24H_CARDINAL'}
    value_cols = [c for c in df_avail.columns if c not in {ID_COL, TIME_COL} and c not in meta_cols]
    df_qc01 = df_avail.sort_values([ID_COL, TIME_COL]).copy()
    df_qc01[f'DVAT_{param}'] = df_qc01.duplicated(subset=[ID_COL] + value_cols, keep='first')
    before = len(df_qc01)
    df_dvat = df_qc01.drop_duplicates(subset=[ID_COL] + value_cols, keep='first').reset_index(drop=True)
    after = len(df_dvat)
    logging.info(f"Removed {before - after} duplicate rows.")
    df_dvat.groupby(ID_COL)[f'DVAT_{param}'].sum().reset_index().to_csv(os.path.join(sum_dir, '01.Summary_DVAT.csv'), index=False)
    df_dvat = df_dvat.drop_duplicates(subset=[ID_COL, TIME_COL], keep='first').reset_index(drop=True)

    save_qc_step(
        df_dvat, param, valid_1991, valid_1981,
        adj_dir, step_dirs['01.DuplicatesRemoved'],
        '01.DVAT_removed', 'After Duplicate Removal'
    )

    df_qc02 = df_dvat.copy()
    if f'QC_{param}' not in df_qc02.columns:
        df_qc02[f'QC_{param}'] = df_qc02[f'RAW_{param}'].copy()
    if param in TEMP_COLS:
        for temp_col in TEMP_COLS:
            if f'QC_{temp_col}' not in df_qc02.columns:
                df_qc02[f'QC_{temp_col}'] = df_qc02[f'RAW_{temp_col}'].copy()

    # Langkah 3: Consistency Check
    logging.info(f"Langkah 3: Consistency check untuk {param}...")
    df_qc03 = check_consistency(df_qc02, param, AVG_COL, MIN_COL, MAX_COL, ID_COL, TIME_COL, sum_dir)
    save_qc_step(
        df_qc03, param, valid_1991, valid_1981,
        adj_dir, step_dirs['02.ConsistencyCheck'],
        '02.consistency_checked', 'After consistency check'
    )

    # Langkah 4: Range Check
    logging.info(f"Langkah 4: Range check untuk {param}...")
    df_qc04 = check_range(df_qc03, param, PHYSICAL_BOUNDS, ID_COL, sum_dir)
    save_qc_step(
        df_qc04, param, valid_1991, valid_1981,
        adj_dir, step_dirs['03.RangeCheck'],
        '03.range_checked', 'After range check'
    )

    # Langkah 5: Abrupt Change Adjustment
    logging.info(f"Langkah 5: Abrupt change adjustment untuk {param}...")
    threshold = ABRUPT_THRESHOLDS.get(param, 2.5)
    df_qc05 = adjust_for_abrupt_changes(df_qc04, param, threshold, ID_COL, TIME_COL)
    save_qc_step(
        df_qc05, param, valid_1991, valid_1981,
        adj_dir, step_dirs['04.AbruptChangesAdjusted'],
        '05.abrupt_adjusted', 'After abrupt change adjustment'
    )
    # =================================================================
    # LANGKAH 6 & 7: EXTEND DATA HUJAN DENGAN ROBI + PLOT PERBANDINGAN (HANYA UNTUK HUJAN)
    # =================================================================
    if param == 'RAINFALL_24H_MM':
        logging.info("Langkah 6: Memperluas data hujan dengan ROBI (1991–2020) per stasiun...")
        robi_raw_col = 'RAW_RAINFALL_24H_MM_ROBI'
        robi_qc_col  = 'QC_RAINFALL_24H_MM_ROBI'
        extend_col   = 'QC_RAINFALL_24H_MM_ROBI_EXTEND'
        # Pastikan kolom ROBI ada
        if robi_raw_col in df_qc05.columns:
            # Salin ROBI ke QC jika belum ada
            if robi_qc_col not in df_qc05.columns:
                df_qc05[robi_qc_col] = df_qc05[robi_raw_col].copy()
            # Inisialisasi kolom EXTEND dengan data QC utama
            df_qc05[extend_col] = df_qc05['QC_RAINFALL_24H_MM'].copy()
            # Proses per stasiun: timpa dengan ROBI di periode 1991–2020
            def extend_per_station(group):
                # Mask hanya untuk periode 1991–2020 (inklusif)
                mask_1991_2020 = (
                    (group[TIME_COL] >= pd.to_datetime('1991-01-01')) &
                    (group[TIME_COL] <= pd.to_datetime('2020-12-31'))
                )
                
                # Salin nilai QC sebagai dasar
                extended_values = group['QC_RAINFALL_24H_MM'].copy()
                
                # Di periode 1991–2020: ganti dengan ROBI jika ada
                robi_values = group[robi_qc_col]
                extended_values.loc[mask_1991_2020] = robi_values.loc[mask_1991_2020].fillna(
                    extended_values.loc[mask_1991_2020]
                )
                
                group[extend_col] = extended_values
                return group
            df_qc05 = df_qc05.groupby(ID_COL, group_keys=False).apply(extend_per_station)
            logging.info(f"✅ Kolom {extend_col} dibuat per stasiun.")
        else:
            logging.warning("Kolom RAW_RAINFALL_24H_MM_ROBI tidak ditemukan. Gunakan data QC saja.")
            df_qc05[extend_col] = df_qc05['QC_RAINFALL_24H_MM'].copy()
        # Simpan juga untuk subset 80%
        for baseline, valid_list in [('1991', valid_1991), ('1981', valid_1981)]:
            df_subset = df_qc05[df_qc05[ID_COL].isin(valid_list)].copy()
            start_baseline = '1981-01-01' if baseline == '1981' else '1991-01-01'
            df_subset = df_subset[df_subset[TIME_COL] >= pd.to_datetime(start_baseline)]
            #output_path = os.path.join(adj_dir, f'06.rainfall_extended_{baseline}.csv')
            output_path = os.path.join(FINAL_DIR, f'{param}_extended_{baseline}.csv')
            df_subset.to_csv(output_path, index=False)
            logging.info(f"Subset ≥80% (baseline {baseline}) dengan data extended disimpan: {output_path}")
        # =================================================================
        # LANGKAH 7: PLOT PERBANDINGAN
        # =================================================================
        logging.info("Langkah 7: Membuat plot perbandingan data hujan sebelum dan sesudah ekstensi...")
        plot_dir = os.path.join(main_out, '06.RainfallExtended', 'plots')
        os.makedirs(plot_dir, exist_ok=True)
        # Gabungkan daftar stasiun ≥80%
        valid_stations_combined = list(set(valid_1981) | set(valid_1991))
        for sid in valid_stations_combined:
            try:
                sdf = df_qc05[df_qc05[ID_COL] == sid].copy()
                if sdf.empty:
                    continue
                plt.figure(figsize=(12, 5))
                plt.plot(sdf[TIME_COL], sdf['QC_RAINFALL_24H_MM'], color='gray', alpha=0.6, label='QC Asli')
                plt.plot(sdf[TIME_COL], sdf[extend_col], color='darkblue', linewidth=1.2, label='Hybrid (ROBI 1991–2020 + QC)')
                plt.axvspan(pd.to_datetime('1991-01-01'), pd.to_datetime('2020-12-31'), color='orange', alpha=0.1, label='Periode ROBI (1991–2020)')
                plt.title(f'Perbandingan Data Hujan – Stasiun {sid}', fontsize=14)
                plt.xlabel('Tanggal')
                plt.ylabel('Curah Hujan (mm)')
                plt.legend()
                plt.grid(True, linestyle='--', alpha=0.6)
                plt.tight_layout()
                plot_path = os.path.join(plot_dir, f'Station_{sid}_rainfall_extend_comparison.png')
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                plt.close()
            except Exception as e:
                logging.warning(f"Gagal buat plot untuk stasiun {sid}: {e}")
                continue
        
        logging.info(f"✅ Plot perbandingan disimpan di: {plot_dir}")
    
    # Akhir blok khusus hujan
    DF_FINAL_DICT[param] = df_qc05.copy()

# Simpan file final gabungan
for param, df_final in DF_FINAL_DICT.items():
    final_dir  = os.path.join(QC_DIR, param, '06.Adjusted')
    final_path = os.path.join(final_dir, f'{param}_FINAL_QC_DATA_LEVEL1.csv')
    df_final.to_csv(final_path, index=False)
    logging.info(f"File final disimpan: {final_path}")

# ====== FUNGSI MUAT DATA ======
def load_availability_data():
    all_records = []
    for param in PARAMS:
        summary_file = os.path.join(QC_DIR, param, '05.Summary', '00.Summary_80percent.csv')
        if not os.path.exists(summary_file):
            logging.warning(f"File tidak ditemukan: {summary_file}")
            continue
        df = pd.read_csv(summary_file)
        logging.info(f"Membaca summary kelengkapan: {param} ({len(df)} stasiun)")
        def safe_subset(avail_col, meets_col, baseline):
            if avail_col in df.columns and meets_col in df.columns:
                subset = df[['WMO_ID', avail_col, meets_col]].copy()
                subset.columns = ['WMO_ID', 'availability', 'meets_80pct']
                subset['param'] = param
                subset['baseline'] = baseline
                
                # Konversi availability ke numerik
                subset['availability'] = pd.to_numeric(subset['availability'], errors='coerce')
                
                # Konversi meets_80pct dengan aman
                def to_bool(val):
                    if pd.isna(val):
                        return False
                    if isinstance(val, bool):
                        return val
                    if isinstance(val, (int, float)):
                        return bool(val)
                    if isinstance(val, str):
                        return val.strip().lower() in ('true', '1', 'yes')
                    return False
                subset['meets_80pct'] = subset['meets_80pct'].apply(to_bool)
                # Hapus baris dengan availability NaN
                subset = subset.dropna(subset=['availability'])
                return subset
            return None
        # Baseline 1981
        subset_1981 = safe_subset(
            f'AVAIL_{param}_1981_2026',
            f'80%_{param}_1981',
            '1981'
        )
        if subset_1981 is not None:
            print(f"{param} 1981: {subset_1981['meets_80pct'].sum()} stasiun memenuhi")
            all_records.append(subset_1981)
        # Baseline 1991
        subset_1991 = safe_subset(
            f'AVAIL_{param}_1991_2026',
            f'80%_{param}_1991',
            '1991'
        )
        if subset_1991 is not None:
            print(f"{param} 1991: {subset_1991['meets_80pct'].sum()} stasiun memenuhi")
            all_records.append(subset_1991)
    if all_records:
        return pd.concat(all_records, ignore_index=True)
    else:
        return pd.DataFrame(columns=['WMO_ID', 'availability', 'meets_80pct', 'param', 'baseline'])
avail = load_availability_data()
# =================================================================
# EKSTRAK DATA TERPISAH QC LEVEL 2: HANYA STASIUN ≥80% BASELINE 1981 DAN 1991
# =================================================================
logging.info("\nMemisahkan data akhir per baseline (1981 dan 1991)...")
for param in PARAMS:
    final_path = os.path.join(QC_DIR, param, '06.Adjusted', f'{param}_FINAL_QC_DATA_LEVEL1.csv')
    if not os.path.exists(final_path):
        logging.warning(f"File final tidak ditemukan: {final_path}. Lewati.")
        continue
    df_final = pd.read_csv(final_path, parse_dates=[TIME_COL])
    # Hitung ulang daftar stasiun ≥80%
    valid_1981 = avail[(avail['param'] == param) & (avail['baseline'] == '1981') & (avail['meets_80pct'])]['WMO_ID'].tolist()
    valid_1991 = avail[(avail['param'] == param) & (avail['baseline'] == '1991') & (avail['meets_80pct'])]['WMO_ID'].tolist()
    logging.info(f"{param}: {len(valid_1981)} stasiun lolos ≥80% untuk baseline 1981.")
    logging.info(f"{param}: {len(valid_1991)} stasiun lolos ≥80% untuk baseline 1991.")
    # Simpan terpisah
    for baseline, valid_list in [('1981', valid_1981), ('1991', valid_1991)]:
        df_subset = df_final[df_final[ID_COL].isin(valid_list)].copy()
        if baseline == '1981':
            start_date = '1981-01-01'
        else:
            start_date = '1991-01-01'
        df_subset = df_subset[df_subset[TIME_COL] >= pd.to_datetime(start_date)]
        if len(df_subset) == 0:
            logging.warning(f"Tidak ada stasiun lolos ≥80% untuk {param} baseline {baseline}.")
            continue
        output_file = os.path.join(
            QC_DIR, param, '06.Adjusted',
            f'{param}_FINAL_QC_DATA_LEVEL1_80PCT_ONLY_{baseline}.csv'
        )
        df_subset.to_csv(output_file, index=False)
        logging.info(f"Subset baseline {baseline} disimpan: {output_file} ({len(valid_list)} stasiun)")
logging.info("Pipeline QC selesai. File final dan subset ≥80% tersedia.")

2026-01-27 12:15:44,641 - INFO - Memulai pipeline QC untuk parameter suhu dan curah hujan...
2026-01-27 12:15:44,643 - INFO - Loading raw data from: data/00.Raw_Dataset/02.BMKGSOFT_VIEW_FKLIM_DAILY_1991-2024_UPDATED_20260127.csv
2026-01-27 12:15:53,473 - INFO - Kolom berikut ditemukan: ['RAINFALL_24H_MM']
2026-01-27 12:15:54,911 - INFO - Duplikat berdasarkan waktu dihapus: 214889 baris
2026-01-27 12:15:55,027 - INFO - 
2026-01-27 12:15:55,028 - INFO - Memproses parameter: RAINFALL_24H_MM
2026-01-27 12:15:55,029 - INFO - ============================================================
2026-01-27 12:15:56,996 - INFO - Langkah 1: Menghitung kelengkapan untuk baseline 1981 dan 1991...
2026-01-27 12:15:57,302 - INFO - [1991_2026] Jumlah stasiun: 189 → 101 lolos (≥80%)
2026-01-27 12:15:57,825 - INFO - [1981_2026] Jumlah stasiun: 190 → 91 lolos (≥80%)
2026-01-27 12:18:27,726 - INFO - Langkah 2: Removing duplicates for RAINFALL_24H_MM...
2026-01-27 12:18:32,634 - INFO - Removed 150129 duplicate ro

RAINFALL_24H_MM 1981: 91 stasiun memenuhi
RAINFALL_24H_MM 1991: 101 stasiun memenuhi


/tmp/ipykernel_1447280/130484495.py:504: DtypeWarning: Columns (1,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv(final_path, parse_dates=[TIME_COL])
2026-01-27 12:35:01,629 - INFO - RAINFALL_24H_MM: 91 stasiun lolos ≥80% untuk baseline 1981.
2026-01-27 12:35:01,632 - INFO - RAINFALL_24H_MM: 101 stasiun lolos ≥80% untuk baseline 1991.
2026-01-27 12:35:28,821 - INFO - Subset baseline 1981 disimpan: data/01.QC_Dataset_Level_01/RAINFALL_24H_MM/06.Adjusted/RAINFALL_24H_MM_FINAL_QC_DATA_LEVEL1_80PCT_ONLY_1981.csv (91 stasiun)
2026-01-27 12:35:52,665 - INFO - Subset baseline 1991 disimpan: data/01.QC_Dataset_Level_01/RAINFALL_24H_MM/06.Adjusted/RAINFALL_24H_MM_FINAL_QC_DATA_LEVEL1_80PCT_ONLY_1991.csv (101 stasiun)
2026-01-27 12:35:52,666 - INFO - Pipeline QC selesai. File final dan subset ≥80% tersedia.
